In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
# read the data file
df = pd.read_csv(
    "/content/labeled_jobs (1).csv",
    encoding="utf-8-sig"
)

In [ ]:
# basic info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5700 entries, 0 to 5699
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         5700 non-null   int64 
 1   job_text   5700 non-null   object
 2   gpt_label  5700 non-null   object
dtypes: int64(1), object(2)
memory usage: 133.7+ KB


In [ ]:
df['gpt_label'].unique()

array(['Other_Irrelevant', 'Advanced Data Science & Research',
       'AI & Tech Product Management', 'IT administration & support',
       'AI & Machine Learning Engineering',
       'Data Engineering & Architecture',
       'Data Governance, Privacy & Management',
       'DevOps, SRE & Systems Administration',
       'Data Analytics & Business Intelligence',
       'Software & Application Development',
       'Cybersecurity & Information Security',
       'Cloud Engineering & Architecture'], dtype=object)

In [ ]:
df['gpt_label'].value_counts()

,count
gpt_label,
Other_Irrelevant,1403
Data Analytics & Business Intelligence,976
"DevOps, SRE & Systems Administration",705
Cybersecurity & Information Security,451
Software & Application Development,408
Cloud Engineering & Architecture,397
"Data Governance, Privacy & Management",329
IT administration & support,296
AI & Tech Product Management,257


# Classification

In [ ]:
X = df["job_text"]
y = df["gpt_label"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

model.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer(max_features=5000)),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, f1_score

print(classification_report(y_test, y_pred))

f1_macro = f1_score(y_test, y_pred, average="macro")
print("F1 Macro:", f1_macro)

                                        precision    recall  f1-score   support

     AI & Machine Learning Engineering       0.77      0.65      0.71        26
          AI & Tech Product Management       0.47      0.81      0.59        52
      Advanced Data Science & Research       0.73      0.96      0.83        25
      Cloud Engineering & Architecture       0.83      0.67      0.74        79
  Cybersecurity & Information Security       1.00      0.88      0.93        90
Data Analytics & Business Intelligence       0.90      0.75      0.82       195
       Data Engineering & Architecture       0.77      0.91      0.83        44
 Data Governance, Privacy & Management       0.71      0.73      0.72        66
  DevOps, SRE & Systems Administration       0.80      0.74      0.77       141
           IT administration & support       0.52      0.80      0.63        59
                      Other_Irrelevant       0.87      0.79      0.83       281
    Software & Application Development 

In [ ]:
!pip install transformers datasets evaluate -q
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df["job_text"].tolist()
y = df["gpt_label"].tolist()

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=512)

In [ ]:
import torch

class JobDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = JobDataset(train_encodings, y_train)
test_dataset = JobDataset(test_encodings, y_test)

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(le.classes_)
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=50
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.119733,0.899459
2,0.617449,0.658307
3,0.438690,0.561072
4,0.386558,0.536091


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1140, training_loss=0.7788191778617993, metrics={'train_runtime': 1116.4472, 'train_samples_per_second': 16.338, 'train_steps_per_second': 1.021, 'total_flos': 2416636247408640.0, 'train_loss': 0.7788191778617993, 'epoch': 4.0})

In [ ]:
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=1)

from sklearn.metrics import f1_score, classification_report
import numpy as np

# Extract true labels from the test_dataset used for prediction
# This ensures y_test is numerical and consistent with the model's training
true_labels_from_dataset = np.array([item['labels'].item() for item in test_dataset])

print(classification_report(true_labels_from_dataset, y_pred, target_names=le.classes_))

print("F1 Macro:", f1_score(true_labels_from_dataset, y_pred, average="macro"))

                                        precision    recall  f1-score   support

     AI & Machine Learning Engineering       0.77      0.65      0.71        26
          AI & Tech Product Management       0.69      0.65      0.67        52
      Advanced Data Science & Research       0.74      0.80      0.77        25
      Cloud Engineering & Architecture       0.78      0.77      0.78        79
  Cybersecurity & Information Security       0.98      0.97      0.97        90
Data Analytics & Business Intelligence       0.90      0.90      0.90       195
       Data Engineering & Architecture       0.84      0.82      0.83        44
 Data Governance, Privacy & Management       0.80      0.80      0.80        66
  DevOps, SRE & Systems Administration       0.79      0.83      0.81       141
           IT administration & support       0.69      0.63      0.65        59
                      Other_Irrelevant       0.89      0.88      0.89       281
    Software & Application Development 

In [ ]:
!pip freeze > requirements.txt

In [ ]:
SAVE_PATH = "./job_classifier_model"

id2label = {i: label for i, label in enumerate(le.classes_)}
label2id = {label: i for i, label in enumerate(le.classes_)}

model.config.id2label = id2label
model.config.label2id = label2id

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./job_classifier_model/tokenizer_config.json',
 './job_classifier_model/tokenizer.json')

In [ ]:
from google.colab import files
files.download('job_classifier_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>